# 复现回测结果

验证本地回测与 `--from-predictions` 结果一致。

In [6]:
import os, sys, pickle, json
import pandas as pd
import numpy as np
sys.path.insert(0, '../code/src')
from backtest import ETFBacktester, run_backtest, run_backtest_from_predictions
import warnings
warnings.filterwarnings('ignore')

## 配置

In [7]:
MODEL_DIR = "../model/search_itransformer_74_3/exp_54"
MODEL_FILE = "best_model_sliding.pth"
DATA_PATH = "../etf_data/etf_74.csv"
CACHE_DIR = "../output/predictions_cache"
BT_CACHE_DIR = "../output/backtest_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(BT_CACHE_DIR, exist_ok=True)

START_DATE = "2026-04-01"
END_DATE = "2026-05-12"
TOP_K = 3
REBALANCE_DAYS = 5
POSITION_PCT = 0.95
INITIAL_CAPITAL = 100000

In [ ]:
import glob
from tqdm import tqdm

BASE_DIR = "../model"
MODEL_TYPES = [
                "bayes_itransformer_74_3",
                "search_itransformer_74_3",
                "bayes_dlinear_74_3",
                "bayes_lstm_74_3",
                "bayes_gru_74_3",
                "search_tcn_74_3",
               ]

EXPERIMENTS = []
for mt in MODEL_TYPES:
    cnt = 0
    for exp_dir in sorted(glob.glob(f"{BASE_DIR}/{mt}/exp_*")):
        if os.path.exists(f"{exp_dir}/best_model_sliding.pth"):
            EXPERIMENTS.append((exp_dir, "best_model_sliding.pth"))
        if os.path.exists(f"{exp_dir}/best_model.pth"):
            EXPERIMENTS.append((exp_dir, "best_model.pth"))
        if os.path.exists(f"{exp_dir}/best_model_ndcg.pth"):
            EXPERIMENTS.append((exp_dir, "best_model_ndcg.pth"))
        cnt += 1
    print(f"{mt}: 共 {cnt} 个实验，找到 {len(EXPERIMENTS)} 个模型文件")
print(f"共 {len(EXPERIMENTS)} 个实验")

bayes_itransformer_74_3: 共 162 个实验，找到 324 个模型文件
search_itransformer_74_3: 共 72 个实验，找到 468 个模型文件
bayes_dlinear_74_3: 共 87 个实验，找到 642 个模型文件
bayes_lstm_74_3: 共 0 个实验，找到 642 个模型文件
bayes_gru_74_3: 共 162 个实验，找到 964 个模型文件
search_tcn_74_3: 共 54 个实验，找到 1070 个模型文件
共 1070 个实验


## 遍历所有实验，一次性缓存 + 回测

In [9]:
import time

cached_data, cached_features = ETFBacktester.load_data_once(
    data_path=DATA_PATH,
    scaler_path=f'{MODEL_DIR}/scaler.pkl',
    feature_num='39',
    verbose=True,
)


使用缓存数据: ../etf_data/etf_74.csv


In [ ]:


all_results = []
for exp_dir, mf in tqdm(EXPERIMENTS, desc="回测"):
    cache_key = f"{exp_dir}/{mf}"
    safe_name = cache_key.replace("\\", "/").replace('../', '').replace('./', '').replace('/', '_')
    cache_path = os.path.join(CACHE_DIR, f"{safe_name}.pkl")
    
    if not os.path.exists(cache_path):
        try:
            bt = ETFBacktester.from_cached_data(
                model_dir=exp_dir, cached_data=cached_data,
                cached_features=cached_features, device='cpu',
                model_file=mf, verbose=False,
            )
            preds = bt.generate_predictions_dict(start_date=START_DATE, end_date=END_DATE, rebalance_days=REBALANCE_DAYS, first_rebalance_date=START_DATE)
            with open(cache_path, 'wb') as f:
                pickle.dump(preds, f)
            del bt.model, bt
        except Exception as e:
            print(f'FAIL gen {cache_key}: {e}')
            continue
    else:
        with open(cache_path, 'rb') as f:
            preds = pickle.load(f)
    
    for mode in ['close', 'open']:
        bt_cache_key = f"{safe_name}_{mode}.pkl"
        bt_cache_path = os.path.join(BT_CACHE_DIR, bt_cache_key)
        if os.path.exists(bt_cache_path):
            with open(bt_cache_path, 'rb') as f:
                row = pickle.load(f)
            all_results.append(row)
            continue
        try:
            r = run_backtest_from_predictions(
                predictions_dict=preds, data_path=DATA_PATH,
                start_date=START_DATE, end_date=END_DATE,
                top_k=TOP_K, rebalance_days=REBALANCE_DAYS,
                position_pct=POSITION_PCT, initial_capital=INITIAL_CAPITAL,
                trade_mode=mode, verbose=False, log=False,
            )
            row = {
                'experiment': exp_dir.replace("\\", "/").split('/')[-2] + '/' + exp_dir.replace("\\", "/").split('/')[-1],
                'model_file': mf, 'trade_mode': mode,
                'return': r.strategy_return,
                'dd': r.max_drawdown,
                'hs300': r.hs300_return,
                'excess': r.excess_return,
                'win_rate': r.rebalance_stats.get('win_rate', 0),
                'avg_return': r.rebalance_stats.get('avg_return', 0),
                'rebalances': r.rebalance_stats.get('total', 0),
            }
            with open(bt_cache_path, 'wb') as f:
                pickle.dump(row, f)
            all_results.append(row)
        except Exception as e:
            print(f'FAIL backtest {cache_key} {mode}: {e}')

df = pd.DataFrame(all_results)
for mode in ['close', 'open']:
    sub = df[df['trade_mode'] == mode].sort_values('return', ascending=False).head(5)
    print(f'\n=== {mode} Top 5 ===')
    for _, r in sub.iterrows():
        print(f'  {r["experiment"]:35s} {r["model_file"]:25s} return={r["return"]:6.2f}%  dd={r["dd"]:5.2f}%  win={r["win_rate"]:5.1f}%  avg={r["avg_return"]:+5.2f}%')

回测:   0%|          | 0/1070 [00:00<?, ?it/s]

回测:  83%|████████▎ | 893/1070 [16:11<02:35,  1.14it/s]  

In [12]:
df = pd.DataFrame(all_results)
for mode in ['close', 'open']:
    sub = df[df['trade_mode'] == mode].sort_values('avg_return', ascending=False).head(10)
    display(sub[['experiment', 'model_file', 'return', 'avg_return','dd', 'hs300', 'excess']])

,experiment,model_file,return,avg_return,dd,hs300,excess
952,bayes_dlinear_74_3/exp_12,best_model_sliding.pth,31.57,4.63,3.20,9.53,22.04
1136,bayes_dlinear_74_3/exp_54,best_model_sliding.pth,31.57,4.63,3.20,9.53,22.04
1140,bayes_dlinear_74_3/exp_55,best_model_sliding.pth,30.35,4.39,3.20,9.53,20.82
1152,bayes_dlinear_74_3/exp_58,best_model_sliding.pth,30.35,4.39,3.20,9.53,20.82
1132,bayes_dlinear_74_3/exp_53,best_model_sliding.pth,29.46,4.22,3.11,9.53,19.93
320,bayes_itransformer_74_3/exp_25,best_model_sliding.pth,27.61,4.20,5.88,9.53,18.08
184,bayes_itransformer_74_3/exp_14,best_model_sliding.pth,26.84,4.19,5.09,9.53,17.31
186,bayes_itransformer_74_3/exp_14,best_model.pth,26.84,4.19,5.09,9.53,17.31
500,bayes_itransformer_74_3/exp_66,best_model_sliding.pth,28.42,4.10,5.88,9.53,18.89
984,bayes_dlinear_74_3/exp_2,best_model_sliding.pth,28.77,4.09,2.79,9.53,19.24


,experiment,model_file,return,avg_return,dd,hs300,excess
1133,bayes_dlinear_74_3/exp_53,best_model_sliding.pth,19.10,6.10,1.14,9.53,9.57
1141,bayes_dlinear_74_3/exp_55,best_model_sliding.pth,19.10,6.10,1.14,9.53,9.57
953,bayes_dlinear_74_3/exp_12,best_model_sliding.pth,19.10,6.10,1.14,9.53,9.57
1153,bayes_dlinear_74_3/exp_58,best_model_sliding.pth,19.10,6.10,1.14,9.53,9.57
985,bayes_dlinear_74_3/exp_2,best_model_sliding.pth,15.46,5.50,1.69,9.53,5.93
987,bayes_dlinear_74_3/exp_2,best_model.pth,15.46,5.50,1.69,9.53,5.93
961,bayes_dlinear_74_3/exp_14,best_model_sliding.pth,16.36,5.31,0.97,9.53,6.84
1137,bayes_dlinear_74_3/exp_54,best_model_sliding.pth,15.23,4.97,1.14,9.53,5.70
1025,bayes_dlinear_74_3/exp_29,best_model_sliding.pth,14.02,4.59,0.97,9.53,4.49
981,bayes_dlinear_74_3/exp_19,best_model_sliding.pth,12.22,3.93,1.43,9.53,2.69
